# Bronze Layer — Raw Data Ingestion

**Medallion Stage:** Bronze (Raw)

This notebook covers the first stage of the Medallion architecture:
loading the raw IBM HR Attrition dataset, profiling it without modifications,
and persisting it to the bronze layer with a metadata manifest.

> **No transformations happen here.** Bronze = raw, immutable truth.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path so src/ imports work
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import json
from IPython.display import display

# Plot aesthetics
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
sns.set_palette('Set2')
print(f'Working root: {ROOT}')

## 1. Load Raw CSV

Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` into `data/bronze/` before running.
Download: https://www.kaggle.com/datasets/patelprashant/employee-attrition

In [ ]:
RAW_CSV = ROOT / 'data' / 'bronze' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv'

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Raw CSV not found at {RAW_CSV}\n'
        'Download from Kaggle and place it at data/bronze/WA_Fn-UseC_-HR-Employee-Attrition.csv'
    )

df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns')

## 2. Schema Overview

In [ ]:
dtype_df = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.values,
    'Non-Null': df.notna().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().mean() * 100).round(2).values,
    'Unique Values': df.nunique().values,
    'Sample': [df[c].dropna().iloc[0] if df[c].notna().any() else 'N/A' for c in df.columns],
})
display(dtype_df)

## 3. Missingness Map

This dataset is known to be complete — the matrix below confirms it.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
msno.bar(df, ax=ax, color='steelblue', fontsize=9)
ax.set_title('Completeness per Column', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

null_total = df.isnull().sum().sum()
print(f'Total null cells: {null_total}  ({"No missing data" if null_total == 0 else "MISSING DATA FOUND"})')

## 4. Constant Columns

Three columns carry the same value for every row — they are artefacts of the dataset
generation and will be removed in the Silver layer.

In [ ]:
constants = [col for col in df.columns if df[col].nunique() == 1]
print(f'Constant columns ({len(constants)}):')
for col in constants:
    print(f'  {col:30s} = {df[col].iloc[0]!r}')

## 5. Duplicate Check

In [ ]:
dup_rows = df.duplicated().sum()
dup_emp  = df.duplicated(subset=['EmployeeNumber']).sum()
print(f'Duplicate full rows       : {dup_rows}')
print(f'Duplicate EmployeeNumbers : {dup_emp}')
assert dup_emp == 0, 'Duplicate employee IDs detected — investigate before proceeding'

## 6. Target Variable Distribution

In [ ]:
counts = df['Attrition'].value_counts()
rate   = counts['Yes'] / len(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Attrition Count', fontweight='bold')
axes[0].set_ylabel('Employees')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Attrition Proportion', fontweight='bold')

plt.suptitle(f'Overall Attrition Rate: {rate:.1%}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Attrition rate: {rate:.1%}  |  Class imbalance ratio: 1 : {(1-rate)/rate:.1f}')

## 7. Numerical Feature Summary

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
desc = df[num_cols].describe().T
desc['skew'] = df[num_cols].skew().round(3)
desc['kurt'] = df[num_cols].kurtosis().round(3)
display(desc.style.background_gradient(subset=['mean', 'std', 'skew'], cmap='RdYlGn_r'))

## 8. Categorical Value Counts

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Attrition')  # already covered above

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Count')
    for j, v in enumerate(vc.values):
        axes[i].text(v + 5, j, str(v), va='center', fontsize=9)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Save Bronze Layer

In [ ]:
from src.utils.config import load_config
from src.bronze.ingest import BronzeIngester

cfg = load_config()
ingester = BronzeIngester(cfg)
bronze_df = ingester.ingest(RAW_CSV)

# Verify metadata
meta_path = ROOT / 'data' / 'bronze' / 'metadata.json'
with open(meta_path) as f:
    meta = json.load(f)

print('Bronze metadata saved:')
for k, v in meta.items():
    if k not in ('columns', 'dtypes', 'null_counts'):
        print(f'  {k}: {v}')

---
## Summary

| Item | Value |
|---|---|
| Rows | 1,470 |
| Columns | 35 |
| Missing values | 0 |
| Constant columns | 3 (EmployeeCount, Over18, StandardHours) |
| Attrition rate | ~16.1% |
| Class imbalance | 1 : 5.2 |

**Next step:** `02_silver_data_cleaning.ipynb` — clean, encode, and enrich.